# 08 · Failures, checkpoints and goodput

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/distributed/08_checkpoint_failures.ipynb) &nbsp; 📖 Concept deck: [Reliability and stability](https://nilesh-salpe.github.io/visual-ai/distributed/11-reliability-stability/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** Young's checkpoint interval, a Monte Carlo simulation of a training run with random failures, and the effect of faster (async, in-memory) checkpoints on effective training time.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

HOURS = 24 * 54                       # a 54-day run
MTBF = 54 * 24 / 419                  # Llama 3 405B: 419 interruptions in 54 days → ~3.1 h
RESTART = 10 / 60                     # 10 minutes to detect, restart and reload

## Task 1: Young's approximation

If a checkpoint costs C hours and failures arrive every M hours on average, the interval that minimizes lost time is approximately
$$\tau^\star \approx \sqrt{2\,C\,M}$$
(Young, 1974). Return τ* in hours.

In [ ]:
def young_interval(C, M):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_young_interval(f):
    assert np.isclose(f(5 / 60, MTBF), np.sqrt(2 * 5 / 60 * MTBF))
    assert np.isclose(f(0.5, 2.0), np.sqrt(2.0))

check("young_interval", _test_young_interval, young_interval)

## Task 2: Simulate a run

Simulate wall-clock time until `HOURS` of useful training is done:
* work in segments of `tau` hours followed by a checkpoint costing `C` (training is paused while it writes);
* failures arrive as a Poisson process with mean gap `M` (draw exponential gaps with `rng.exponential(M)`); a failure loses all work since the last completed checkpoint and costs `RESTART`.

Return **goodput** = useful hours ÷ wall-clock hours. Use the given `rng` so results are reproducible.

In [ ]:
def simulate_goodput(tau, C, M=MTBF, seed=0):
    rng = np.random.default_rng(seed)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_simulate_goodput(f):
    g_good = f(young_interval_solution(5 / 60, MTBF) if 'young_interval_solution' in globals() else np.sqrt(2 * 5 / 60 * MTBF), 5 / 60)
    g_rare = f(12.0, 5 / 60); g_often = f(0.05, 5 / 60)
    assert 0.5 < g_good < 1.0
    assert g_good > g_rare and g_good > g_often, f"Young's interval ({g_good:.3f}) should beat checkpointing far too rarely ({g_rare:.3f}) or too often ({g_often:.3f})"
    assert np.isclose(f(1.0, 0.01, M=1e9), 1 / 1.01, atol=1e-3), "with no failures, goodput = tau / (tau + C)"

check("simulate_goodput", _test_simulate_goodput, simulate_goodput)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Young's approximation
def young_interval_solution(C, M):
    return np.sqrt(2 * C * M)

check("young_interval (reference)", _test_young_interval, young_interval_solution)

In [ ]:
# @title Solution for Task 2: Simulate a run
def simulate_goodput_solution(tau, C, M=MTBF, seed=0):
    rng = np.random.default_rng(seed)
    useful = wall = 0.0
    next_fail = rng.exponential(M)
    while useful < HOURS:
        seg = min(tau, HOURS - useful)
        if wall + seg + C <= next_fail:          # segment and checkpoint finish before the next failure
            wall += seg + C; useful += seg
        else:                                     # failure: lose the partial segment, pay restart
            wall = next_fail + RESTART
            next_fail = wall + rng.exponential(M)
    return useful / wall

check("simulate_goodput (reference)", _test_simulate_goodput, simulate_goodput_solution)

---
## Experiments
**Faster checkpoints change everything.** Compare a 5-minute synchronous checkpoint with a 30-second one (async write, the GPU only pauses to snapshot) and a 5-second in-memory one (Gemini-style: copies to peer CPU memory, reported >13× faster recovery). Each uses its own Young interval.

In [ ]:
for name, C in [('sync to storage, 5 min', 5 / 60), ('async, 30 s pause', 0.5 / 60), ('in-memory, 5 s', 5 / 3600)]:
    tau = young_interval_solution(C, MTBF)
    g = np.mean([simulate_goodput_solution(tau, C, seed=s) for s in range(5)])
    print(f"{name:24} interval {tau * 60:5.1f} min → goodput {g:.1%}")
print("\nLlama 3 reported >90% effective training time over 54 days on 16,384 GPUs.")

---
## Takeaways
* Failures are routine at scale: what matters is how much work each one costs.
* Checkpoint interval ≈ √(2·C·MTBF); cheaper checkpoints allow more frequent saves and less lost work.
* Async and in-memory checkpointing, fast restart, and spare capacity are the levers that keep goodput above 90%.